In [ ]:
!pip install fastapi uvicorn pyngrok sentence-transformers PyPDF2 faiss-cpu transformers torch -q
!pip install --user streamlit -q
!pkill -f streamlit
!pkill -f uvicorn

In [ ]:
# 1. Load Models
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
print("Loading Mistral-Nemo (this may take a while)...")
model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_llm = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

In [ ]:
import uvicorn
import threading
import time
import socket
import torch
import numpy as np
import faiss
import io
from fastapi import FastAPI, Request, HTTPException, UploadFile, Form
from pyngrok import ngrok, conf
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Configuration ---
NGROK_TOKEN = "ADD YOURS"
API_KEY = "secret123"

app = FastAPI()

# 2. RAG Helper Functions
def generate_text(prompt, max_length=500):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model_llm.generate(**inputs, max_length=max_length, do_sample=True, temperature=0.7)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def chunk_text(text, chunk_size=50, overlap=5):
    words = text.split()
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]

print("Loading Embedding Model...")
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

@app.post("/ask")
async def ask_question(req: Request, file: UploadFile, question: str = Form(...)):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401)
    
    try:
        # Read uploaded file bytes directly from memory
        file_bytes = await file.read()
        reader = PdfReader(io.BytesIO(file_bytes))
        raw_text = "\n".join([page.extract_text() for page in reader.pages])
        
        # Chunk and Index the uploaded document on the fly
        chunks = chunk_text(raw_text)
        embeddings = embed_model.encode(chunks, convert_to_numpy=True)
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(embeddings)
        
        # Search the vector index
        query_embedding = embed_model.encode([question], convert_to_numpy=True)
        _, indices = index.search(query_embedding, k=1)
        context_chunk = chunks[indices[0][0]]
        
        # Prompt LLM
        prompt = f"""[INST] You are a helpful assistant. Answer the user's question based ONLY on the provided reference text. 
        Reference Text:
        {context_chunk}
        Question: {question} [/INST]"""
        
        answer = generate_text(prompt)
        
        return {"question": question, "answer": answer, "context": context_chunk}
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def free_port():
    s = socket.socket(); s.bind(('', 0))
    p = s.getsockname()[1]; s.close()
    return p

api_port = free_port()
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=api_port), daemon=True).start()
time.sleep(5)
print(f"RAG API ready on port: {api_port}")

In [ ]:
streamlit_code = f"""
import streamlit as st
import requests

st.set_page_config(page_title="Custom PDF AI Assistant", page_icon="📄")
st.title("Ask Questions about the PDF(Mistral-Nemo)")

API_URL = "http://127.0.0.1:{api_port}/ask"
API_KEY = "secret123"

uploaded_file = st.file_uploader("Upload a PDF document:", type=["pdf"])

with st.form("question_form", clear_on_submit=False):
    question = st.text_input("Ask a question about this document:")
    submit_button = st.form_submit_button("Get Answer")

if submit_button:
    if not uploaded_file:
        st.warning("Please upload a PDF document first.")
    elif not question:
        st.warning("Please type a question.")
    else:
        with st.spinner("Mistral is reading your document and thinking..."):
            try:
                files = {{"file": (uploaded_file.name, uploaded_file.getvalue(), "application/pdf")}}
                payload = {{"question": question}}
                headers = {{"Authorization": f"Bearer {{API_KEY}}"}}
                
                resp = requests.post(API_URL, files=files, data=payload, headers=headers)
                
                if resp.status_code == 200:
                    data = resp.json()
                    raw_answer = data.get('answer', '').strip()
                    
                    # Safely skip past the literal "Answer" text and its leading symbols if present
                    if raw_answer.startswith("Answer"):
                        clean_answer = raw_answer[6:].lstrip("r\\\\n ").lstrip("n ").strip()
                    else:
                        clean_answer = raw_answer
                    
                    st.write("**Answer:**")
                    st.write(clean_answer)
                else:
                    st.error(f"API Error (Status {{resp.status_code}}). Check backend logs.")
            except Exception as e:
                st.error(f"Connection failed: {{e}}")
"""
with open("streamlit_app.py", "w") as f:
    f.write(streamlit_code)

In [ ]:
import subprocess
import sys
import os
import time
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN

try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
except:
    pass

try:
    public_url = ngrok.connect(8501).public_url
    print(f"✅ RAG UI Link: {public_url}")
except Exception as e:
    print(f"❌ ngrok error: {e}")

if os.path.exists("streamlit_app.py"):
    proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "streamlit_app.py", "--server.port", "8501"])
    print(f"🚀 Streamlit started (PID: {proc.pid})")
else:
    print("❌ Error: streamlit_app.py not found! Run Cell 3 first.")